In [50]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [51]:
df = pd.read_csv('../data/analysed_data.csv')
df.head()

,order_status,customer_state,price,freight_value,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,...,payment_value,customer_lat,customer_lng,seller_lat,seller_lng,delivery_time_days,approval_hours,carrier_handover_hours,purchase_month,purchase_dayofweek
0,delivered,SP,29.99,8.72,4.0,500.0,19.0,8.0,13.0,housewares,...,18.12,-23.576983,-46.587161,-23.680729,-46.444238,15,0.178333,56.795833,10,0
1,delivered,SP,29.99,8.72,4.0,500.0,19.0,8.0,13.0,housewares,...,2.00,-23.576983,-46.587161,-23.680729,-46.444238,15,0.178333,56.795833,10,0
2,delivered,SP,29.99,8.72,4.0,500.0,19.0,8.0,13.0,housewares,...,18.59,-23.576983,-46.587161,-23.680729,-46.444238,15,0.178333,56.795833,10,0
3,delivered,BA,118.70,22.76,1.0,400.0,19.0,13.0,19.0,perfumery,...,141.46,-12.177924,-44.660711,-19.807681,-43.980427,19,30.713889,11.109167,7,1
4,delivered,GO,159.90,19.22,1.0,420.0,24.0,19.0,21.0,auto,...,179.12,-16.745150,-48.514783,-21.363502,-48.229601,26,0.276111,4.910278,8,2


In [52]:
# Check for missing values
df.isnull().sum()

order_status                     0
customer_state                   0
price                            0
freight_value                    0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_category_name_english    0
seller_state                     0
payment_type                     0
payment_installments             0
payment_value                    0
customer_lat                     0
customer_lng                     0
seller_lat                       0
seller_lng                       0
delivery_time_days               0
approval_hours                   0
carrier_handover_hours           0
purchase_month                   0
purchase_dayofweek               0
dtype: int64

In [53]:
# Check for duplicates
print("Number of duplicate rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("Number of duplicate rows after dropping:", df.duplicated().sum())

Number of duplicate rows: 1
Number of duplicate rows after dropping: 0


# Feature Engineering

In [54]:
# Created a new feature distance between customer and seller using the Euclidean distance formula
df['distance'] = np.sqrt((df['customer_lat'] - df['seller_lat'])**2 + (df['customer_lng'] - df['seller_lng'])**2)

In [55]:
# Larger products takes more time to deliver, so we can create a new feature for product volume
df['product_volume'] = (df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm'])

In [56]:
# Orders from the same state might be delivered faster, so we created a new feature same_state
df['same_state'] = (df['customer_state'] == df['seller_state']).astype(int)

In [57]:
# Expensive shipping relative to product price often indicates longer-distance deliveries.
df['freight_ratio'] = (df['freight_value'] / (df['price'] + 1))

In [58]:
# Product density can also be a factor in delivery time, as denser products may require special handling.
df['product_density'] = (df['product_weight_g'] / (df['product_volume'] + 1))

In [59]:
# Created a new feature for the total size of the package, which can be calculated as the sum of length, height, and width.
df['package_size'] = (df['product_length_cm'] + df['product_height_cm'] + df['product_width_cm'])

In [60]:
df.drop(columns=['customer_lat', 'customer_lng', 'seller_lat', 'seller_lng'], inplace=True)

In [61]:
df.dtypes

order_status                         str
customer_state                       str
price                            float64
freight_value                    float64
product_photos_qty               float64
product_weight_g                 float64
product_length_cm                float64
product_height_cm                float64
product_width_cm                 float64
product_category_name_english        str
seller_state                         str
payment_type                         str
payment_installments             float64
payment_value                    float64
delivery_time_days                 int64
approval_hours                   float64
carrier_handover_hours           float64
purchase_month                     int64
purchase_dayofweek                 int64
distance                         float64
product_volume                   float64
same_state                         int64
freight_ratio                    float64
product_density                  float64
package_size    

In [62]:
data = df.to_csv("../data/processed_data.csv", index=False)